In [1]:
from Bio import SeqIO
import re

In [2]:
gffile      = "../../../../assembly_12B1/ncbi/GCA_041296205.1_12B1_v1.1_genomic.gff"       
protfile = "../../../../assembly_12B1/ncbi/GCA_041296205.1_12B1_v1.1_protein.faa"      
lookupfile = '../../../../assembly_12B1/12B1_geneid_ncbi_lookup.txt'
fafile  = "SV36_proteins.fasta"
eggfile = '../../../../hybrid_diversity/7_functional_enrichment/02_eggnog/02_eggnog.emapper.annotations'
eggoutfile = 'SV36_eggknog_annotations.txt'
annfile = '../../../../genome_diversity/6_Genome_annotation/7_Functional_Annotations/all_annotations_kinfin.txt'
annoutfile = 'SV36_kinfin_annotations.txt'

chrom  = "CM083880.1"                 
range_start   = 324000                  
range_end     = 434000                  


In [3]:
genedict = {}
with open(gffile, 'r') as fi:
    for line in fi:
        col = line.rstrip().split('\t')
        if col[0] == chrom and col[2] == 'gene':
        
            gene_left  = int(col[3])
            gene_right = int(col[4])
            
            if gene_left <= range_end and gene_right >= range_start:
                locus_tag = col[8].split('locus_tag=')[1]
                strand = col[6]
                print(gene_left, gene_right, strand, locus_tag)
                genedict[locus_tag] = [chrom, str(gene_left), str(gene_right), strand]

322275 325205 - AB1Y20_012930
325523 327566 + AB1Y20_012931
329219 340513 + AB1Y20_012932
340863 341813 + AB1Y20_012933
344566 346440 + AB1Y20_012934
356516 357297 + AB1Y20_012935
358373 361312 - AB1Y20_012936
361453 362649 - AB1Y20_012937
362840 364480 + AB1Y20_012938
364642 366143 + AB1Y20_012939
367196 367723 + AB1Y20_012940
375913 376631 - AB1Y20_012941
378393 379344 - AB1Y20_012942
386743 387110 - AB1Y20_012943
387923 388744 - AB1Y20_012944
388907 391503 + AB1Y20_012945
392396 393435 + AB1Y20_012946
396141 397039 - AB1Y20_012947
398410 400690 - AB1Y20_012948
402053 403075 - AB1Y20_012949
403678 404674 + AB1Y20_012950
408061 415242 - AB1Y20_012951
418080 419017 + AB1Y20_012952
423409 426652 + AB1Y20_012953
428141 428494 - AB1Y20_012954
429333 434586 - AB1Y20_012955


In [4]:
with open(lookupfile, 'r') as fi:
    for line in fi:
        [oldid, newid, note, protid, locus] = line.split('\t')
        locus = locus.rstrip()
        if locus in genedict:
            #print(line)
            genedict[locus] = [newid, oldid] + genedict[locus]

In [5]:
protdict = {}
with open(fafile, "w") as fo:
    for record in SeqIO.parse(protfile, "fasta"):
        match = re.search(r'[A-Z0-9]+_\d+', record.description)
        locus_tag = match.group() if match else None        
        
        if locus_tag in genedict:
            protdict[record.id] = genedict[locus_tag]
            
            new_header = re.sub(r'hypothetical protein |(\[Prymnesium parvum\])', '', record.description)
            new_header += f'[{" ".join(genedict[locus_tag])}]'
            print(new_header)
            
            fo.write(f'>{new_header}\n{record.seq}\n')

KAL1500263.1 AB1Y20_012930 [12B1s25g229880 12B1g22988 CM083880.1 322275 325205 -]
KAL1500264.1 AB1Y20_012931 [12B1s25g229890 12B1g22989 CM083880.1 325523 327566 +]
KAL1500265.1 AB1Y20_012932 [12B1s25g229900 12B1g22990 CM083880.1 329219 340513 +]
KAL1500266.1 AB1Y20_012933 [12B1s25g229910 12B1g22991 CM083880.1 340863 341813 +]
KAL1500267.1 AB1Y20_012934 [12B1s25g229920 12B1g22992 CM083880.1 344566 346440 +]
KAL1500268.1 AB1Y20_012935 [12B1s25g229930 12B1g22993 CM083880.1 356516 357297 +]
KAL1500269.1 AB1Y20_012936 [12B1s25g229940 12B1g22994 CM083880.1 358373 361312 -]
KAL1500270.1 AB1Y20_012937 [12B1s25g229950 12B1g22995 CM083880.1 361453 362649 -]
KAL1500271.1 AB1Y20_012938 [12B1s25g229960 12B1g22996 CM083880.1 362840 364480 +]
KAL1500272.1 AB1Y20_012939 [12B1s25g229970 12B1g22997 CM083880.1 364642 366143 +]
KAL1500273.1 AB1Y20_012940 [12B1s25g229980 12B1g22998 CM083880.1 367196 367723 +]
KAL1500274.1 AB1Y20_012941 [12B1s25g229990 12B1g22999 CM083880.1 375913 376631 -]
KAL1500275.1 AB1

In [6]:
with open(eggfile, 'r') as fi:
    for line in fi:
        col = line.rstrip().split('\t')
        #print(col[0])
        protid = col[0]
        if protid in protdict:
            #print(protdict[protid], col)
            protdict[protid] = protdict[protid] + col

In [7]:
with open(eggoutfile, 'w') as fo:
    fo.write("gene id\told gene id\tchromosome\tgene start\tgene end\tstrand\tprotein accn\tseed_ortholog\t")
    fo.write("evalue\tscore\teggNOG_OGs\tmax_annot_lvl\tCOG_category\tDescription\tPreferred_name\t")
    fo.write("GOs\tEC\tKEGG_ko\tKEGG_Pathway\tKEGG_Module\tKEGG_Reaction\tKEGG_rclass\tBRITE\tKEGG_TC\tCAZy\tBiGG_Reaction\tPFAMs\n")
    
    for protid in protdict:
        annline = '\t'.join(protdict[protid])
        if len(protdict[protid]) > 25:
            fo.write(f'{annline}\n')
        else:
            fo.write(f'{annline}\t{protid}\n')
        

In [8]:
newprotdict = {}
for protid in protdict:
    oldid = protdict[protid][1]
    protdict[protid][1] = protid
    newprotdict[oldid] = protdict[protid][:6]

In [9]:
with open(annfile, 'r') as fi:
    for line in fi:
        col = line.rstrip().split('\t')
        col = ['' if x == 'None' else x for x in col]
        #print(col[0])
        protid = col[0]
        if protid.split('_')[1] == '12B1':
            newid = '12B1' + protid.split('.')[0]
            #print(newid)
            if newid in newprotdict:
                #print(newprotdict[newid], col)
                newprotdict[newid] = newprotdict[newid] + col

In [10]:
with open(annoutfile, 'w') as fo:
    #fo.write("gene id\told gene id\tchromosome\tgene start\tgene end\tstrand\tprotein accn\tseed_ortholog\t")
    #fo.write("evalue\tscore\teggNOG_OGs\tmax_annot_lvl\tCOG_category\tDescription\tPreferred_name\t")
    #fo.write("GOs\tEC\tKEGG_ko\tKEGG_Pathway\tKEGG_Module\tKEGG_Reaction\tKEGG_rclass\tBRITE\tKEGG_TC\tCAZy\tBiGG_Reaction\tPFAMs\n")
    
    for protid in newprotdict:
        annline = '\t'.join(newprotdict[protid])
        fo.write(f'{annline}\n')
       
        